# 05 · Model comparison & error analysis

Consolidates everything logged by notebooks 02–04 into the deliverables the roadmap asks for:

* `results/model_comparison.csv` – the Table 5 template of the paper, now populated
* Per-class F1 heatmap across models/conditions
* Confusion matrix + most-confused pairs for the **final model**
* Error slices: short text, negation, possible sarcasm cues, source (GoEmotions vs Dreaddit), confidence
* Minority-class misclassification examples (`results/reports/error_examples.csv`)

**Needs:** `results/model_metrics.csv`, `results/predictions_*.csv`, optionally `models/final_model.json`.

In [ ]:
!pip -q install scikit-learn seaborn

In [ ]:
import os, sys, json, random, time, warnings, re, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

# ---- Project root: Google Drive folder on Colab, repo root locally ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/mental-health-nlp")   # <- change if you keep the repo elsewhere
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

for d in ["data/raw", "data/interim", "data/processed", "data/splits",
          "models/classical", "models/bilstm", "models/distilbert", "models/roberta",
          "results/figures", "results/reports"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

SEED = 42
EMOTIONS = ["Joy", "Sadness", "Anger", "Fear", "Anxiety", "Stress", "Surprise", "Neutral"]
LABEL2ID = {e: i for i, e in enumerate(EMOTIONS)}
ID2LABEL = {i: e for e, i in LABEL2ID.items()}
NEGATIVE = ["Sadness", "Anger", "Fear", "Anxiety", "Stress"]      # N in the paper (Sec. 5.6)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_seed()
print("ROOT =", ROOT)

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             confusion_matrix, classification_report)

METRICS_CSV = ROOT / "results/model_metrics.csv"
LATENCY_CSV = ROOT / "results/latency_results.csv"
LABEL_IDS = list(range(len(EMOTIONS)))

def slug(condition):
    """File-system friendly condition name ('augment+class_weight' -> 'augment-class_weight')."""
    return condition.replace("+", "-")

def _upsert(path, row, keys):
    df = pd.read_csv(path) if path.exists() else pd.DataFrame()
    if len(df):
        mask = np.ones(len(df), dtype=bool)
        for k in keys:
            mask &= (df[k].astype(str) == str(row[k])).values
        df = df[~mask]
    pd.concat([df, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)

def evaluate(y_true, y_pred, model, condition="baseline", seed=SEED, extra=None, log=True):
    """Compute all metrics, optionally upsert into results/model_metrics.csv."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABEL_IDS,
                                                 average="macro", zero_division=0)
    per = f1_score(y_true, y_pred, labels=LABEL_IDS, average=None, zero_division=0)
    row = dict(model=model, condition=condition, seed=seed,
               accuracy=accuracy_score(y_true, y_pred),
               precision_macro=p, recall_macro=r, macro_f1=f, min_class_f1=per.min())
    row.update({f"f1_{e}": v for e, v in zip(EMOTIONS, per)})
    if extra:
        row.update(extra)
    if log:
        _upsert(METRICS_CSV, row, ["model", "condition", "seed"])
    return row

def log_latency(model, device, ms_per_text, note=""):
    _upsert(LATENCY_CSV, dict(model=model, device=device, ms_per_text=round(float(ms_per_text), 2), note=note),
            ["model", "device", "note"])

def plot_cm(y_true, y_pred, title, path=None, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_IDS)
    if normalize:
        cm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7.5, 6))
    sns.heatmap(cm, annot=True, fmt=".2f" if normalize else "d", cmap="Blues",
                xticklabels=EMOTIONS, yticklabels=EMOTIONS, ax=ax, cbar=False)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=150)
    plt.show()

## 1 · Model comparison table (mean ± std over seeds)

In [ ]:
m = pd.read_csv(METRICS_CSV)
if "val_macro_f1" not in m: m["val_macro_f1"] = np.nan
agg = (m.groupby(["model", "condition"])
         .agg(accuracy=("accuracy", "mean"), macro_f1=("macro_f1", "mean"), macro_f1_std=("macro_f1", "std"),
              min_class_f1=("min_class_f1", "mean"), val_macro_f1=("val_macro_f1", "mean"),
              n_seeds=("seed", "nunique")).reset_index())
agg["macro_f1_std"] = agg["macro_f1_std"].fillna(0)

lat = pd.read_csv(LATENCY_CSV) if LATENCY_CSV.exists() else pd.DataFrame(columns=["model", "device", "ms_per_text"])
lat_cpu = lat[lat.device == "cpu"].groupby("model")["ms_per_text"].min()

ORDER = ["logreg", "svm", "rf", "bilstm", "distilbert", "roberta"]
rows = []
for mdl in [o for o in ORDER if o in set(agg.model)]:
    g = agg[agg.model == mdl]
    base = g[g.condition == "baseline"]
    best = g.sort_values("val_macro_f1", ascending=False).iloc[0]              # best mitigation chosen on validation
    for r in ([base.iloc[0]] if len(base) else []) + ([best] if (len(base) == 0 or best.condition != "baseline") else []):
        rows.append(dict(Model=mdl, Condition=r.condition, Accuracy=r.accuracy, **{"Macro-F1": r.macro_f1,
                         "Macro-F1 std": r.macro_f1_std, "Min-class F1": r.min_class_f1,
                         "Latency CPU (ms)": lat_cpu.get(mdl, np.nan), "Seeds": int(r.n_seeds)}))
comparison = pd.DataFrame(rows)
comparison.round(4).to_csv(ROOT / "results/model_comparison.csv", index=False)
display(comparison.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
b = comparison.copy(); b["label"] = b["Model"] + "\n(" + b["Condition"] + ")"
ax.bar(b["label"], b["Macro-F1"], yerr=b["Macro-F1 std"], color=sns.color_palette("deep")[0], capsize=3)
ax.set_ylabel("Test macro-F1"); ax.set_title("Model comparison (error bars = std over seeds)")
plt.xticks(rotation=0, fontsize=8); plt.tight_layout()
plt.savefig(ROOT / "results/figures/05_model_comparison.png", dpi=150); plt.show()

pc = (m.groupby(["model", "condition"])[[f"f1_{e}" for e in EMOTIONS]].mean()
        .rename(columns=lambda c: c[3:]))
fig, ax = plt.subplots(figsize=(9, 0.42 * len(pc) + 1.5))
sns.heatmap(pc, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1, ax=ax, cbar_kws={"label": "F1"})
ax.set_title("Per-class test F1 — every model × condition"); ax.set_ylabel("")
plt.tight_layout(); plt.savefig(ROOT / "results/figures/05_per_class_f1_heatmap.png", dpi=150); plt.show()

## 2 · Pick the final model and load its test predictions

In [ ]:
meta = ROOT / "models/final_model.json"
if meta.exists():
    info = json.loads(meta.read_text()); FM, FC = info["model"], info["condition"]
else:
    r = agg.sort_values("val_macro_f1", ascending=False).iloc[0]; FM, FC = r.model, r.condition
print(f"Final model: {FM} | condition: {FC}")

P = pd.read_csv(ROOT / f"results/predictions_{FM}_{slug(FC)}.csv")
P["n_words"] = P["text"].astype(str).str.split().str.len()
y_true = P["true"].map(LABEL2ID).values; y_pred = P["pred"].map(LABEL2ID).values
print(classification_report(y_true, y_pred, target_names=EMOTIONS, digits=3, zero_division=0))
plot_cm(y_true, y_pred, f"Confusion matrix — {FM} ({FC})", path=ROOT / "results/figures/05_cm_final_model.png")

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=LABEL_IDS)
pairs = [(EMOTIONS[i], EMOTIONS[j], cm[i, j], cm[i, j] / cm[i].sum()) for i in range(8) for j in range(8) if i != j]
pairs = pd.DataFrame(pairs, columns=["true", "predicted_as", "count", "share_of_true_class"]).sort_values("count", ascending=False)
print("Most frequent confusions:")
display(pairs.head(10).round(3))

## 3 · Error slices

In [ ]:
P["correct"] = P["true"] == P["pred"]
NEG_RE = re.compile(r"\b(not|no|never|nothing|nobody|cannot)\b|n't", re.I)
SARC_RE = re.compile(r"yeah right|oh great|as if|thanks a lot|/s\b|sure,? because|love how|nice job|great job|oh wow|totally|obviously", re.I)

slices = {
    "all": P,
    "very short (<=5 words)": P[P.n_words <= 5],
    "short (6-15)": P[(P.n_words > 5) & (P.n_words <= 15)],
    "long (>15)": P[P.n_words > 15],
    "contains negation": P[P.text.astype(str).str.contains(NEG_RE)],
    "possible sarcasm cue (heuristic)": P[P.text.astype(str).str.contains(SARC_RE)],
}
for s in P.source.unique():
    slices[f"source = {s}"] = P[P.source == s]

def slice_f1(g):
    labs = sorted(set(g.true.map(LABEL2ID)))
    return f1_score(g.true.map(LABEL2ID), g.pred.map(LABEL2ID), labels=labs, average="macro", zero_division=0)

sl = pd.DataFrame([dict(slice=k, n=len(g), accuracy=g.correct.mean(), macro_f1=slice_f1(g)) for k, g in slices.items() if len(g)])
display(sl.round(3))

In [ ]:
# Dreaddit-confound check: does the model simply call every LONG text "Stress"?
long_ = P[P.n_words > 40]
print("Texts >40 words:", len(long_), "| share predicted Stress:", round((long_.pred == "Stress").mean(), 3),
      "| share truly Stress:", round((long_.true == "Stress").mean(), 3))
gm = P[P.source == "goemotions"]
print("GoEmotions texts predicted Stress:", round((gm.pred == "Stress").mean(), 4),
      "(if Stress is tied to length/domain, expect GoEmotions texts to rarely receive it)")

# Calibration / confidence (supports the 'don't act on low-confidence predictions' future-work idea)
if P["conf"].notna().any():
    P["conf_bin"] = pd.cut(P["conf"], [0, .4, .6, .8, .9, 1.0])
    display(P.groupby("conf_bin").agg(n=("correct", "size"), accuracy=("correct", "mean")).round(3))

## 4 · Minority-class error examples (for the report)

In [ ]:
minority = ["Fear", "Anxiety", "Surprise", "Stress"]
ex = []
for c in minority:
    wrong = P[(P.true == c) & (~P.correct)]
    print(f"\n=== TRUE {c}: {len(wrong)} errors out of {(P.true == c).sum()} ===")
    for _, r in wrong.sample(min(6, len(wrong)), random_state=SEED).iterrows():
        print(f"  pred={r.pred:9s} | {str(r.text)[:150]}")
    ex.append(wrong.sample(min(25, len(wrong)), random_state=SEED))
pd.concat(ex)[["true", "pred", "conf", "source", "text"]].to_csv(ROOT / "results/reports/error_examples.csv", index=False)

In [ ]:
# Highest-confidence mistakes: usually label noise or sarcasm
if P["conf"].notna().any():
    display(P[~P.correct].sort_values("conf", ascending=False).head(10)[["true", "pred", "conf", "text"]].round(3))

### Suggested write-up points (fill in from the tables above)
1. **RQ1** – ranking of classical vs Bi-LSTM vs transformers on macro-F1 (state std over seeds; only claim differences larger than seed noise).
2. **RQ2** – which mitigation helped which class; did majority-class F1 drop?
3. **RQ3** – GoEmotions-style vs Dreaddit-style performance, Stress confusion with Sadness/Anxiety, and the length confound above.
4. **Failure modes** – sarcasm, very short text, negation, Fear↔Anxiety confusion, label noise in high-confidence mistakes.

Next: **`06_temporal_analysis.ipynb`**.